In [1]:
import os 
os.chdir("../")

In [2]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

c:\Users\vedas\anaconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [9]:
extracted_data = load_pdf_files("data")


In [10]:
len(extracted_data)

637

In [11]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
            page_content=doc.page_content,
            metadata={"source": src}
            )
        )
    return minimal_docs

In [12]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [13]:
#chunking 
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap=20,
    )
    texts_chunk= text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [14]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 5859


In [15]:
from langchain.embeddings import HuggingFaceBgeEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings= HuggingFaceBgeEmbeddings(
        model_name=model_name
    )
    return embeddings
embedding = download_embeddings()

C:\Users\vedas\AppData\Local\Temp\ipykernel_31304\1717374157.py:5: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings= HuggingFaceBgeEmbeddings(


In [11]:
embedding

HuggingFaceBgeEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_instruction='Represent this question for searching relevant passages: ', embed_instruction='', show_progress=False)

In [12]:
vector =embedding.embed_query("Hello World")
vector

[-0.010300806723535061,
 0.1830792874097824,
 0.030811214819550514,
 0.004452835768461227,
 -0.027336126193404198,
 -0.0335625596344471,
 0.03763147443532944,
 -0.03157336637377739,
 -0.003391023725271225,
 -0.008950845338404179,
 0.03803614154458046,
 -0.05129106342792511,
 0.0003683052200358361,
 -0.02372710220515728,
 0.09271024167537689,
 -0.02779580093920231,
 -0.03515253961086273,
 -0.003224164480343461,
 -0.0768178179860115,
 -0.05761215090751648,
 0.07257598638534546,
 0.11128546297550201,
 0.016058487817645073,
 0.015908492729067802,
 -0.08232703804969788,
 0.007007284555584192,
 0.029013115912675858,
 0.0011386839905753732,
 0.11671736091375351,
 -0.032327305525541306,
 -0.03227158263325691,
 -0.0012590468395501375,
 0.10591626167297363,
 0.023600861430168152,
 0.009664921090006828,
 0.09834079444408417,
 0.042936377227306366,
 -0.019547628238797188,
 0.019267886877059937,
 -0.06417104601860046,
 0.02392338030040264,
 -0.05288007855415344,
 -0.026469502598047256,
 0.005548716

In [3]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [4]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [5]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key) 

In [16]:
pc

In [6]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,
        metric = "cosine",
        spec = ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [23]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [20]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
    
)

## Add more data

In [ ]:
addex= Document(
    page_content="how to add addtional data",
    metadata= {"source":"example"}
)

In [ ]:
docsearch.add_documents(documents=[addex])

This will generate search key to search on the pinecone platform.

In [21]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})


In [22]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='befba96e-76fe-4628-a4de-9808add28c92', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='26e232e5-dd2e-4dde-b579-0bcc7bf741a0', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='ce7a574b-eb84-43ba-af37-031816c65b07', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Cliffs, NJ: Prentice Hall, 1995.\nGoldstein, Sanford M., and Richard B. Odom. “Skin &\nAppendages: Pustular Disorders.” In Current Medical\nDiagnosis and Treatment, 1996.35th ed. Ed. Stephen\nMcPhee, et al. Stamford: Appleton & Lange, 1995

In [23]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o")


In [25]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [26]:
system_prompt=(
    "You are a Medical assistant for question-answering tasks."
    "Use the following pieces of retrieved context to answer the question."
    "If u don't know the answer, say that you don't know."
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"

)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [27]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [29]:
response = rag_chain.invoke({"input":"what is Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly is a disorder caused by the abnormal release of a chemical from the pituitary gland in the brain, leading to increased growth in bone and soft tissue and various other body disturbances. Gigantism is similar, but typically occurs in children before the closure of growth plates, resulting in abnormal height and growth. Both conditions involve the excess of growth hormone and its effects on the body.
